# LLM-as-a-Judge (Pairwise Debiased)

Position bias 보정을 위해 양방향(A→B, B→A) pairwise 평가를 수행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/project_05

In [ ]:
!pip install -q openai pandas numpy matplotlib seaborn

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "***"

client = OpenAI()

In [ ]:
import pandas as pd

CSV_FILES = [
    "/content/drive/MyDrive/project_05/csv/eval_outputs_ab_v2_final.csv",
    "/content/drive/MyDrive/project_05/csv/eval_outputs_qwen35_ab_final.csv",
]

df_raw = pd.concat([pd.read_csv(f) for f in CSV_FILES], ignore_index=True)
print(df_raw.columns)
df_raw.head()

In [ ]:
# -----------------------------
# 1. DATA PREP
# -----------------------------
import json
import itertools
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

MODELS = [
    "qwen2.5-7b-instruct-base",
    "qwen2.5-7b-instruct-lora",
    "qwen3.5-9b-base",
    "qwen3.5-9b-lora",
]

# pivot 전에 question 따로 보존 (row_id + prompt_type 기준 유니크)
df_question = df_raw[["row_id", "prompt_type", "question", "persona"]].drop_duplicates()

df_pivot = df_raw.pivot_table(
    index=["row_id", "prompt_type"],
    columns="model_name",
    values="model_output",
    aggfunc="first"
).reset_index()
df_pivot.columns.name = None

# question, persona 컬럼 붙이기
df_pivot = df_pivot.merge(df_question, on=["row_id", "prompt_type"], how="left")
df_pivot = df_pivot.dropna(subset=MODELS)

print(df_pivot.shape)
print(df_pivot[["row_id", "prompt_type", "persona", "question"]].head())

In [ ]:
# -----------------------------
# 2. Sampling
# -----------------------------
# persona별로 각각 25개씩 → 총 100개 (pressure 25*2 + friendly 25*2)
def balanced_sample(df, n_per_group=25, random_state=42):
    samples = []
    for persona, grp in df.groupby("persona"):
        median = grp["avg_len"].median()
        easy = grp[grp["avg_len"] <  median]
        hard = grp[grp["avg_len"] >= median]
        samples.append(easy.sample(min(n_per_group, len(easy)), random_state=random_state))
        samples.append(hard.sample(min(n_per_group, len(hard)), random_state=random_state))
    return pd.concat(samples)

df_pivot["avg_len"] = df_pivot[MODELS].astype(str).map(len).mean(axis=1)
df_sample = balanced_sample(df_pivot, n_per_group=25)

print(f"샘플: {len(df_sample)}개 | judge 호출: {len(df_sample) * 6}회")
print(df_sample["persona"].value_counts())

In [ ]:
# -----------------------------
# 3. Judge
# -----------------------------
PAIRWISE_PROMPT = """\
당신은 ICT 직무 면접 답변을 평가하는 전문 평가자입니다.
두 개의 면접관 답변을 비교하여 어느 쪽이 더 우수한지 판단하세요.

## 평가 기준
1. 정확성 - 기술적으로 틀린 내용이 없는가?
2. 실무 활용성 - 실제 면접 상황에서 도움이 되는가?
3. 완결성 - 질문에 충분히 답변하고 있는가?
4. 명확성 - 구조가 명확하고 이해하기 쉬운가?

## 규칙
- 점수(숫자)는 절대 사용하지 마세요.
- 오직 A와 B만 비교하세요 (무승부 없음).

## 출력 형식 (JSON만)
{{"winner": "A" 또는 "B", "reason": "간단한 이유 (최대 2문장)"}}

지원자 질문: {question}
답변 A: {answer_a}
답변 B: {answer_b}
"""

def call_judge(prompt):
    response = client.chat.completions.create(
        model="gpt-5-2025-08-07",
        messages=[
            {"role": "system", "content": "당신은 공정하고 엄격한 평가자입니다."},
            {"role": "user",   "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

In [ ]:
# -----------------------------
# 4. Pairwise 실행
# -----------------------------
def run_pairwise(df, models, client):
    pairs = list(itertools.combinations(models, 2))  # 6쌍
    results, cache = [], {}

    for model_a, model_b in pairs:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{model_a} vs {model_b}"):
            key = (model_a, model_b, row["row_id"], row["prompt_type"])

            if key not in cache:
                prompt = PAIRWISE_PROMPT.format(
                    question=row["question"],   # 🔥 실제 question 텍스트
                    answer_a=row[model_a],
                    answer_b=row[model_b]
                )
                output = call_judge(prompt)
                try:
                    w = json.loads(output)["winner"]
                    cache[key] = w if w in ("A", "B") else None
                except Exception as e:
                    print(f"[WARN] {key}: {e} | {output}")
                    cache[key] = None

            results.append({
                "row_id":      row["row_id"],
                "persona":     row["persona"],       # pressure / friendly
                "prompt_type": row["prompt_type"],   # weak / strong
                "model_a":     model_a,
                "model_b":     model_b,
                "winner":      cache[key],
            })

    return pd.DataFrame(results)

In [ ]:
# -----------------------------
# 5. Winrate / Matrix
# -----------------------------
def compute_winrate(results_df, models):
    win, total = {m: 0 for m in models}, {m: 0 for m in models}
    for _, row in results_df.iterrows():
        a, b, w = row["model_a"], row["model_b"], row["winner"]
        total[a] += 1; total[b] += 1
        if w == "A": win[a] += 1
        elif w == "B": win[b] += 1
    return pd.DataFrame([
        {"model": m, "wins": win[m], "total": total[m],
         "win_rate": round(win[m] / max(total[m], 1), 4)}
        for m in models
    ]).sort_values("win_rate", ascending=False)


def build_win_matrix(results_df, models):
    matrix = pd.DataFrame(0, index=models, columns=models)
    for _, row in results_df.iterrows():
        a, b, w = row["model_a"], row["model_b"], row["winner"]
        if w == "A": matrix.loc[a, b] += 1
        elif w == "B": matrix.loc[b, a] += 1
    return matrix

In [ ]:
# -----------------------------
# 1. 샘플 50개로 줄이기
# -----------------------------
def balanced_sample(df, n_per_group=13, random_state=42):  # 13*2*2 ≈ 52개
    samples = []
    for persona, grp in df.groupby("persona"):
        median = grp["avg_len"].median()
        easy = grp[grp["avg_len"] <  median]
        hard = grp[grp["avg_len"] >= median]
        samples.append(easy.sample(min(n_per_group, len(easy)), random_state=random_state))
        samples.append(hard.sample(min(n_per_group, len(hard)), random_state=random_state))
    return pd.concat(samples)

df_pivot["avg_len"] = df_pivot[MODELS].astype(str).map(len).mean(axis=1)
df_sample2 = balanced_sample(df_pivot, n_per_group=13)

print(f"샘플: {len(df_sample2)}개 | judge 호출: {len(df_sample2) * 12}회")
print(df_sample2["persona"].value_counts())

In [ ]:
# -----------------------------
# 2. 양방향 pairwise
# -----------------------------
def run_pairwise_debiased(df, models, client, save_dir):
    pairs_forward  = list(itertools.combinations(models, 2))
    pairs_reversed = [(b, a) for a, b in pairs_forward]
    all_pairs = pairs_forward + pairs_reversed  # 12쌍

    results, cache = [], {}
    checkpoint_path = f"{save_dir}/pairwise_results_debiased_checkpoint.csv"

    for pair_idx, (model_a, model_b) in enumerate(all_pairs):
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"[{pair_idx+1}/12] {model_a} vs {model_b}"):
            key = (model_a, model_b, row["row_id"], row["prompt_type"])

            if key not in cache:
                prompt = PAIRWISE_PROMPT.format(
                    question=row["question"],
                    answer_a=row[model_a],
                    answer_b=row[model_b]
                )
                output = call_judge(prompt)
                try:
                    w = json.loads(output)["winner"]
                    cache[key] = w if w in ("A", "B") else None
                except Exception as e:
                    print(f"[WARN] {key}: {e} | {output}")
                    cache[key] = None

            results.append({
                "row_id":      row["row_id"],
                "persona":     row["persona"],
                "prompt_type": row["prompt_type"],
                "model_a":     model_a,
                "model_b":     model_b,
                "winner":      cache[key],
            })

        # 🔥 pair마다 체크포인트 저장
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)
        print(f"  [checkpoint] {pair_idx+1}/12 저장 ({len(results)}행)")

    return pd.DataFrame(results)

In [ ]:
# -----------------------------
# 3. Winrate (양방향 합산)
# -----------------------------
def compute_winrate_debiased(results_df, models):
    win, total = {m: 0 for m in models}, {m: 0 for m in models}
    for _, row in results_df.iterrows():
        a, b, w = row["model_a"], row["model_b"], row["winner"]
        total[a] += 1; total[b] += 1
        if w == "A": win[a] += 1
        elif w == "B": win[b] += 1

    df_result = pd.DataFrame([
        {"model": m, "wins": win[m], "total": total[m],
         "win_rate": round(win[m] / max(total[m], 1), 4)}
        for m in models
    ]).sort_values("win_rate", ascending=False)

    return df_result

In [ ]:
# -----------------------------
# 4. RUN
# -----------------------------
results_df2 = run_pairwise_debiased(df_sample2, MODELS, client, save_dir)

winrate_df2 = compute_winrate_debiased(results_df2, MODELS)
matrix_df2  = build_win_matrix(results_df2, MODELS)

print("\n=== Debiased Winrate ===")
print(winrate_df2)
print("\n=== Winner 분포 (50:50에 가까울수록 bias 제거됨) ===")
print(results_df2["winner"].value_counts(normalize=True))

results_df2.to_csv(f"{save_dir}/pairwise_results_debiased.csv", index=False)
winrate_df2.to_csv(f"{save_dir}/winrate_debiased.csv", index=False)
matrix_df2.to_csv(f"{save_dir}/win_matrix_debiased.csv")
print("✅ 저장 완료")